In [118]:
import os
import re
import math
import pandas as pd
import numpy as np
from datetime import datetime, timezone

In [119]:
df1 =  pd.read_csv(r"/Users/pardeepwalia/Desktop/untitled folder/2025 datasets/bi_analyst.csv",usecols=['company', 'displayTitle', 'jobDescription','jobLocationCity','jobLocationState', 'jobTypes/0','jobTypes/1','pubDate','salarySnippet/text'])
df2 =  pd.read_csv(r"/Users/pardeepwalia/Desktop/untitled folder/2025 datasets/data_analyst_jobs.csv",usecols=['company', 'displayTitle', 'jobDescription','jobLocationCity','jobLocationState', 'jobTypes/0','jobTypes/1','pubDate','salarySnippet/text'])
df3 =  pd.read_csv(r"/Users/pardeepwalia/Desktop/untitled folder/2025 datasets/data_architect_jobs.csv",usecols=['company', 'displayTitle', 'jobDescription','jobLocationCity','jobLocationState', 'jobTypes/0','jobTypes/1','pubDate','salarySnippet/text'])
df4 =  pd.read_csv(r"/Users/pardeepwalia/Desktop/untitled folder/2025 datasets/data_engineer_jobs.csv",usecols=['company', 'displayTitle', 'jobDescription','jobLocationCity','jobLocationState', 'jobTypes/0','jobTypes/1','pubDate','salarySnippet/text'])
df5 =  pd.read_csv(r"/Users/pardeepwalia/Desktop/untitled folder/2025 datasets/data_science_job.csv",usecols=['company', 'displayTitle', 'jobDescription','jobLocationCity','jobLocationState', 'jobTypes/0','jobTypes/1','pubDate','salarySnippet/text'])
df6 =  pd.read_csv(r"/Users/pardeepwalia/Desktop/untitled folder/2025 datasets/ml_engineer_jobs.csv",usecols=['company', 'displayTitle', 'jobDescription','jobLocationCity','jobLocationState', 'jobTypes/0','jobTypes/1','pubDate','salarySnippet/text'])

In [120]:
df1.columns


Index(['company', 'displayTitle', 'jobDescription', 'jobLocationCity',
       'jobLocationState', 'jobTypes/0', 'jobTypes/1', 'pubDate',
       'salarySnippet/text'],
      dtype='object')

In [121]:
df2.columns

Index(['company', 'displayTitle', 'jobDescription', 'jobLocationCity',
       'jobLocationState', 'jobTypes/0', 'jobTypes/1', 'pubDate',
       'salarySnippet/text'],
      dtype='object')

In [122]:
df3.columns

Index(['company', 'displayTitle', 'jobDescription', 'jobLocationCity',
       'jobLocationState', 'jobTypes/0', 'jobTypes/1', 'pubDate',
       'salarySnippet/text'],
      dtype='object')

In [123]:
df4.columns

Index(['company', 'displayTitle', 'jobDescription', 'jobLocationCity',
       'jobLocationState', 'jobTypes/0', 'jobTypes/1', 'pubDate',
       'salarySnippet/text'],
      dtype='object')

In [124]:
df5.columns

Index(['company', 'displayTitle', 'jobDescription', 'jobLocationCity',
       'jobLocationState', 'jobTypes/0', 'jobTypes/1', 'pubDate',
       'salarySnippet/text'],
      dtype='object')

In [125]:
df6.dtypes

company               object
displayTitle          object
jobDescription        object
jobLocationCity       object
jobLocationState      object
jobTypes/0            object
jobTypes/1            object
pubDate                int64
salarySnippet/text    object
dtype: object

In [126]:
#  Canonical schema

MASTER_COLS = [
    'title','company_name','job_location','via','description',
    'extensions','description_tokens','job_basket','listed_time',
    'location_bucket','seniority_level','salary_mid_annual','SQL',
    'Python','Excel','Tableau','Power BI','Cloud','Cloud_list',
    'skill_buckets','skill_buckets_str','skill_details',
    'skill_details_str','skills_flat','salary_min_annual',
    'salary_max_annual','salary_currency','comp_type','source_file',
    'listed_year'
]


# Skills extraction

def extract_skills(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # unified text
    title_s = df.get('title', '').fillna('').astype(str)
    desc_s  = df.get('description', '').fillna('').astype(str)

    tokens_col = df.get('description_tokens')
    if tokens_col is not None:
        tokens_s = tokens_col.apply(lambda x: ' '.join(x) if isinstance(x, list)
                                              else ('' if pd.isna(x) else str(x)))
    else:
        tokens_s = pd.Series([''] * len(df), index=df.index)

    text_s = (title_s + ' ' + desc_s + ' ' + tokens_s).str.lower()

    # cloud sub-skills (expanded)
    cloud_rx = {
        # Providers
        'aws'       : r'\b(?:aws|amazon web services)\b|(?<!\w)(?:s3|glue|athena|redshift)\b',
        'azure'     : r'\b(?:azure)\b|(?<!\w)(?:synapse|data\s*factory|adf)\b',
        'gcp'       : r'\b(?:gcp|google cloud)\b|(?<!\w)(?:bigquery|dataflow|pub/?sub|pubsub)\b',
        # Data platforms / engines
        'snowflake' : r'\b(?:snowflake)\b',
        'redshift'  : r'\b(?:redshift)\b',
        'bigquery'  : r'\b(?:bigquery)\b',
        'databricks': r'\b(?:databricks)\b',
        'spark'     : r'\b(?:apache\s+spark|pyspark|spark\s*sql)\b',
        # Orchestration / ELT / Streaming
        'airflow'   : r'\b(?:airflow|apache\s*airflow)\b',
        'dbt'       : r'\b(?:dbt|data build tool)\b',
        'kafka'     : r'\b(?:kafka|apache\s*kafka)\b',
        # Infra
        'docker'    : r'\b(?:docker)\b',
        'kubernetes': r'\b(?:kubernetes|k8s)\b',
        'terraform' : r'\b(?:terraform|iac|infrastructure as code)\b',
    }

    # top-level buckets (Cloud derived from cloud_rx)
    rx = {
        'SQL'     : r'\b(?:sql|mysql|postgres(?:ql)?|tsql|oracle\s+sql|snowflake)\b',
        'Python'  : r'\b(?:python|pandas|numpy|scikit-?learn|sklearn)\b',
        'Excel'   : r'\b(?:excel|v\s*look\s*up|vlookup|pivot(?:\s*table)?s?)\b',
        'Tableau' : r'\b(?:tableau)\b',
        'Power BI': r'\b(?:power\s*-?\s*bi|pbi)\b',
    }

    # vectorized matches
    skill_df = pd.DataFrame(
        {k: text_s.str.contains(pat, regex=True, na=False) for k, pat in rx.items()},
        index=df.index
    )

    cloud_df = pd.DataFrame(
        {k: text_s.str_contains(pat, regex=True, na=False) if hasattr(text_s, "str_contains")
         else text_s.str.contains(pat, regex=True, na=False)
         for k, pat in cloud_rx.items()},
        index=df.index
    )

    skill_df['Cloud'] = cloud_df.any(axis=1)
    skill_df['Cloud_list'] = cloud_df.apply(lambda r: [k for k, v in r.items() if v], axis=1)

    # drop stale columns & join flags
    to_drop = [
        'SQL','Python','Excel','Tableau','Power BI','Cloud','Cloud_list',
        'skill_buckets','skill_buckets_str','skill_details','skill_details_str','skills_flat'
    ]
    df = df.drop(columns=[c for c in to_drop if c in df.columns], errors='ignore')
    df = df.join(skill_df)

    # bucket list
    bucket_cols = ['SQL','Python','Excel','Tableau','Power BI','Cloud']
    present_mask = df[bucket_cols].fillna(False)
    long = present_mask.stack()
    long = long[long].rename('present').reset_index()
    skills_list = long.groupby('level_0')['level_1'].agg(list)

    empty_lists = pd.Series([[]] * len(df), index=df.index)
    skills_list = skills_list.reindex(df.index).combine_first(empty_lists)

    df['skill_buckets'] = skills_list
    df['skill_buckets_str'] = df['skill_buckets'].apply(lambda L: ', '.join(L))

    # nested details
    cloud_lists = df['Cloud_list'].apply(lambda L: L if isinstance(L, list) else [])
    df['skill_details'] = cloud_lists.apply(lambda L: {'Cloud': L} if L else {})

    def pretty_details(d: dict) -> str:
        if not d: return ''
        parts = []
        for k, v in d.items():
            parts.append(f"{k}:[{'; '.join(v)}]" if isinstance(v, list) and v else f"{k}:[]")
        return ', '.join(parts)

    df['skill_details_str'] = df['skill_details'].apply(pretty_details)

    # flattened list for counting
    def flatten_skills(i):
        buckets = df.at[i, 'skill_buckets'] if isinstance(df.at[i, 'skill_buckets'], list) else []
        clouds  = df.at[i, 'Cloud_list'] if isinstance(df.at[i, 'Cloud_list'], list) else []
        return list(buckets) + [f"Cloud:{s}" for s in clouds]

    df['skills_flat'] = pd.Index(df.index).map(flatten_skills)

    return df

# Helpers (salary, time, etc.)

def parse_epoch_maybe(x):
    """pubDate is int64; detect sec vs ms; return pandas.Timestamp (UTC) or NaT."""
    try:
        xi = int(x)
        if xi > 10**12:  # ms
            ts = datetime.fromtimestamp(xi / 1000, tz=timezone.utc)
        else:            # seconds
            ts = datetime.fromtimestamp(xi, tz=timezone.utc)
        return pd.to_datetime(ts)
    except Exception:
        return pd.NaT

_money = re.compile(r'(?<!\w)(?:C\$|CA\$|US\$|\$)?\s?([0-9]{2,3}(?:[,\s]?[0-9]{3})*(?:\.[0-9]{1,2})?)', re.I)
per_hour = re.compile(r'\b(hour|hr|hourly)\b', re.I)
per_year = re.compile(r'\b(year|yr|annual|annually|per\s?year)\b', re.I)

def salary_from_text(txt, country_hint=None):
    """Return (min_annual, max_annual, currency, period_guess) from salarySnippet/text."""
    if not isinstance(txt, str) or not txt.strip():
        return (None, None, None, None)
    nums = [float(n.replace(',', '').replace(' ', '')) for n in _money.findall(txt)]
    mn = min(nums) if nums else None
    mx = max(nums) if nums else None

    cur = None
    s = txt.upper()
    if 'CAD' in s or 'C$' in s or 'CA$' in s:
        cur = 'CAD'
    elif 'USD' in s or 'US$' in s:
        cur = 'USD'
    elif '$' in s and country_hint == 'Canada':
        cur = 'CAD'
    elif '$' in s and country_hint == 'United States':
        cur = 'USD'

    if per_year.search(txt):
        period = 'year'
    elif per_hour.search(txt):
        period = 'hour'
    else:
        period = None

    def annualize(a, p):
        if a is None: return None
        if p == 'year': return round(a, 2)
        if p == 'hour': return round(a * 2080, 2)
        return round(a, 2)  # assume already annual

    return (annualize(mn, period), annualize(mx, period), cur, period)
# update
def str_or_empty(x):
    # keep real strings; turn NaN/None into ''; otherwise cast to str
    if isinstance(x, str):
        return x
    if pd.isna(x):
        return ''
    return str(x)
# update
def compose_location(city, state):
    city_s  = str_or_empty(city).strip()
    state_s = str_or_empty(state).strip()
    if city_s and state_s:
        return f"{city_s}, {state_s}"
    return city_s or state_s or None

def infer_location_bucket(text_blob):
    t = (text_blob or "").lower()
    if "remote" in t: return "Remote"
    if "hybrid" in t: return "hybrid"
    if "on-site" in t or "onsite" in t: return "On-site"
    return None

def infer_seniority(title):
    t = (title or "").lower()
    if re.search(r'\b(intern|internship|co[-\s]?op)\b', t): return "Intern"
    if re.search(r'\b(junior|jr\.?|entry[-\s]?level)\b', t): return "Junior"
    if re.search(r'\b(mid[-\s]?level|intermediate|mid)\b', t): return "Mid"
    if re.search(r'\b(senior|sr\.?)\b', t): return "Senior"
    if re.search(r'\b(lead|principal|staff)\b', t): return "Lead/Principal"
    if re.search(r'\b(manager|head|director|vp|vice president)\b', t): return "Manager/Director+"
    return None

def infer_comp_type(ext):
    if not ext: return None
    if isinstance(ext, (list, tuple)):
        txt = " ".join(map(str, ext)).lower()
    else:
        txt = str(ext).lower()
    if "contract" in txt: return "Contract"
    if "full" in txt: return "Full-time"
    if "part" in txt: return "Part-time"
    if "intern" in txt: return "Internship"
    return None

# Standardizer for your schema

def standardize(df_raw: pd.DataFrame, role_label: str, via_label="indeed",
                country_hint=None, source_file=None) -> pd.DataFrame:
    """
    Expects columns:
      company, displayTitle, jobDescription,
      jobLocationCity, jobLocationState,
      jobTypes/0, jobTypes/1, pubDate, salarySnippet/text
    """
    out = pd.DataFrame(index=df_raw.index)

    out['title']         = df_raw.get('displayTitle')
    out['company_name']  = df_raw.get('company')
    out['description']   = df_raw.get('jobDescription')

    city_col  = df_raw.get('jobLocationCity', pd.Series(['']*len(df_raw)))
    state_col = df_raw.get('jobLocationState', pd.Series(['']*len(df_raw)))
    out['job_location'] = [compose_location(c, s) for c, s in zip(city_col, state_col)]

    # Also make "extensions" robust (jobTypes/0, jobTypes/1 might be NaN)
    jt0 = df_raw.get('jobTypes/0', pd.Series(['']*len(df_raw))).apply(str_or_empty)
    jt1 = df_raw.get('jobTypes/1', pd.Series(['']*len(df_raw))).apply(str_or_empty)
    out['extensions'] = pd.concat([jt0, jt1], axis=1).apply(
        lambda r: [x for x in (r.tolist()) if x and x.lower() not in ('nan', 'none')], axis=1
    )

    # fold job types into a simple list
    jt0 = df_raw.get('jobTypes/0').astype(str)
    jt1 = df_raw.get('jobTypes/1').astype(str)
    out['extensions'] = pd.concat([jt0, jt1], axis=1).apply(
        lambda r: [x for x in r if x and x.lower() != 'nan'], axis=1
    )

    out['via']                = via_label
    out['description_tokens'] = [[] for _ in range(len(out))]  # can be filled later
    out['job_basket']         = role_label

    # time
    out['listed_time']        = df_raw.get('pubDate').apply(parse_epoch_maybe)
    out['listed_year']        = pd.to_datetime(out['listed_time'], errors='coerce').dt.year

    # salary
    sal = df_raw.get('salarySnippet/text').apply(lambda t: salary_from_text(t, country_hint))
    out['salary_min_annual'] = [a[0] for a in sal]
    out['salary_max_annual'] = [a[1] for a in sal]
    out['salary_currency']   = [a[2] for a in sal]
    out['salary_mid_annual'] = pd.to_numeric(out[['salary_min_annual','salary_max_annual']].mean(axis=1), errors='coerce')

    # location/seniority buckets
    out['location_bucket'] = [
        infer_location_bucket(" ".join(map(str, x))) for x in zip(out['title'], out['description'], out['extensions'].astype(str))
    ]
    out['seniority_level'] = out['title'].apply(infer_seniority)

    # skills
    out = extract_skills(out)

    # comp type & source_file
    out['comp_type']   = out['extensions'].apply(infer_comp_type)
    out['source_file'] = source_file

    # enforce schema
    for col in MASTER_COLS:
        if col not in out.columns:
            out[col] = None
    out = out[MASTER_COLS].copy()
    return out

# Batch runner

ROLE_FOR = {
    "bi_analyst.csv"         : "Business Intelligence Analyst",
    "data_analyst_jobs.csv"  : "Data Analyst",
    "data_architect_jobs.csv": "Data Architect",
    "data_engineer_jobs.csv" : "Data Engineer",
    "data_science_job.csv"   : "Data Scientist",
    "ml_engineer_jobs.csv"   : "Machine Learning Engineer",
}

# set this to your folder
base_dir = "/Users/pardeepwalia/Desktop/untitled folder/2025 datasets"

def load_and_standardize(file, role_label, via_label="indeed", country_hint=None):
    path = os.path.join(base_dir, file)
    usecols = [
        'company','displayTitle','jobDescription',
        'jobLocationCity','jobLocationState',
        'jobTypes/0','jobTypes/1','pubDate','salarySnippet/text'
    ]
    df_raw = pd.read_csv(path, usecols=usecols)
    return standardize(df_raw, role_label, via_label, country_hint=country_hint, source_file=file)

frames = []
for fname, role in ROLE_FOR.items():
    print(f"processing {fname} as {role}")
    # OPTIONAL: set country_hint per file if you know it (e.g., 'Canada' / 'United States')
    frames.append(load_and_standardize(fname, role))

master = pd.concat(frames, ignore_index=True)

# light de-dupe: title + company + listed_time
dedupe_key = (
    master['title'].fillna('').str.lower() + '|' +
    master['company_name'].fillna('').str.lower() + '|' +
    master['listed_time'].astype(str)
)
master = master.loc[~dedupe_key.duplicated()].reset_index(drop=True)

print("Final master shape:", master.shape)


master.sample(10)

processing bi_analyst.csv as Business Intelligence Analyst
processing data_analyst_jobs.csv as Data Analyst
processing data_architect_jobs.csv as Data Architect
processing data_engineer_jobs.csv as Data Engineer
processing data_science_job.csv as Data Scientist
processing ml_engineer_jobs.csv as Machine Learning Engineer
Final master shape: (1409, 30)


,title,company_name,job_location,via,description,extensions,description_tokens,job_basket,listed_time,location_bucket,...,skill_buckets_str,skill_details,skill_details_str,skills_flat,salary_min_annual,salary_max_annual,salary_currency,comp_type,source_file,listed_year
74,Senior Business Analyst,TD Bank,"Toronto, ON",indeed,"Work Location: Toronto, Ontario, Canada \n \n ...",[],[],Data Analyst,2025-07-14 05:13:20+00:00,None,...,"SQL, Python, Excel, Tableau",{},,"[SQL, Python, Excel, Tableau]",76800.0,115200.0,None,None,data_analyst_jobs.csv,2025
57,Actuarial Analyst I,TD Bank,"Montréal, QC",indeed,"Work Location:\n Montréal, Quebec, Canada\n \...",[],[],Data Analyst,2025-08-19 05:06:40+00:00,None,...,"Python, Excel, Tableau",{},,"[Python, Excel, Tableau]",55400.0,83000.0,None,None,data_analyst_jobs.csv,2025
490,Data Architect,Altitude technology Solutions Inc,"Toronto, ON",indeed,Data Architect with Snowflake and Python exper...,"[Temporary, Contract]",[],Data Architect,2025-08-13 05:00:00+00:00,Remote,...,"SQL, Python, Cloud","{'Cloud': ['aws', 'snowflake', 'kubernetes', '...",Cloud:[aws; snowflake; kubernetes; terraform],"[SQL, Python, Cloud, Cloud:aws, Cloud:snowflak...",100401.6,145600.0,None,Contract,data_architect_jobs.csv,2025
1070,"Data Science Manager, Real-Time Supply Management",Lyft,"Toronto, ON",indeed,"At Lyft, our purpose is to serve and connect. ...",[],[],Data Scientist,2025-05-14 05:20:00+00:00,hybrid,...,SQL,{},,[SQL],136000.0,170000.0,None,None,data_science_job.csv,2025
603,Data Engineer - Senior,Cenergy International Services,"Edmonton, AB",indeed,Description:Project Overview\nThe Government o...,"[Temporary, Contract]",[],Data Engineer,2025-08-22 05:20:00+00:00,Remote,...,"SQL, Python, Power BI, Cloud","{'Cloud': ['aws', 'azure', 'gcp', 'snowflake',...",Cloud:[aws; azure; gcp; snowflake; databricks],"[SQL, Python, Power BI, Cloud, Cloud:aws, Clou...",189280.0,189280.0,None,Contract,data_engineer_jobs.csv,2025
105,Bilingual (French & English) Insights Analyst ...,Confidential,"Delta, BC",indeed,Bilingual (French &amp; English) Insights Anal...,"[Full-time, Permanent]",[],Data Analyst,2025-08-13 04:40:00+00:00,hybrid,...,"Excel, Tableau, Power BI",{},,"[Excel, Tableau, Power BI]",59000.0,59000.0,None,Full-time,data_analyst_jobs.csv,2025
1056,"Data Scientist, Credit Risk",Desjardins,"Montréal, QC",indeed,"As Data Scientist, Credit Risk, you assist wit...",[],[],Data Scientist,2025-08-18 04:06:40+00:00,hybrid,...,"SQL, Python, Power BI",{},,"[SQL, Python, Power BI]",NaN,NaN,None,None,data_science_job.csv,2025
692,"Senior Developer - Infra, Data & AI",CWP Energy,"Montréal, QC",indeed,"At CWP Energy Trading, we are an innovative co...",[],[],Data Engineer,2025-08-08 05:13:20+00:00,None,...,"SQL, Python, Cloud","{'Cloud': ['aws', 'azure', 'gcp', 'redshift', ...",Cloud:[aws; azure; gcp; redshift; bigquery; ai...,"[SQL, Python, Cloud, Cloud:aws, Cloud:azure, C...",NaN,NaN,None,None,data_engineer_jobs.csv,2025
625,Lead Database engineer - VP - C-13,Citigroup,"Mississauga, ON",indeed,About the Role:\n We're looking for a seasone...,[],[],Data Engineer,2025-07-10 04:00:00+00:00,None,...,"SQL, Python, Cloud","{'Cloud': ['spark', 'kafka']}",Cloud:[spark; kafka],"[SQL, Python, Cloud, Cloud:spark, Cloud:kafka]",NaN,NaN,None,None,data_engineer_jobs.csv,2025
927,ML Engineering - ML Ops,Workday,"Vancouver, BC",indeed,Your work days are brighter here.\n \n \n \n ...,[],[],Data Engineer,2025-06-23 03:40:00+00:00,Remote,...,"Python, Cloud","{'Cloud': ['aws', 'databricks', 'spark', 'airf...",Cloud:[aws; databricks; spark; airflow; docker...,"[Python, Cloud, Cloud:aws, Cloud:databricks, C...",122400.0,183600.0,None,None,data_engineer_jobs.csv,2025


In [127]:
# before enrinching job_location
master2 = master.copy()


In [128]:
master2['location_bucket'].value_counts()

location_bucket
hybrid     365
Remote     324
On-site     84
Name: count, dtype: int64

In [129]:
master2['location_bucket'].isna().sum()

np.int64(636)

In [130]:
# Clearing the null values
cond1=master['job_location'].str.contains(
        r'\b(?:canada|toronto|vancouver|ontario|ON|bc|british columbia|quebec|qc|alberta|ab|manitoba|mb|saskatchewan|sk|nova scotia|ns|calgary|montreal|ottawa|PE|YT|NB|NL|NT)\b',case=False,na=False,
        regex=True
    )

cond2 = master['location_bucket'].str.contains('Remote',case=False,regex=False)

cond3=master['job_location'].str.contains(r'\b(?:remote)\b',case=False,na=False,
        regex=True
    )
master2['location_bucket2'] = np.select([cond1.to_numpy(dtype=bool),cond2.to_numpy(dtype=bool),cond3.to_numpy(dtype=bool)],['Canada','Remote','Remote'],default="")

In [131]:
master2['location_bucket2'].value_counts()

location_bucket2
Canada    1269
Remote     132
             8
Name: count, dtype: int64

In [132]:
master['location_bucket']= np.select([cond1.to_numpy(dtype=bool),cond2.to_numpy(dtype=bool),cond3.to_numpy(dtype=bool)],['Canada','Remote','Remote'],default="")

In [133]:
# replaced "" -> np.nan
master['location_bucket'].replace("",np.nan)

0       Canada
1       Canada
2       Canada
3       Canada
4       Canada
         ...  
1404    Canada
1405    Canada
1406    Canada
1407    Canada
1408    Canada
Name: location_bucket, Length: 1409, dtype: object

In [134]:
master['location_bucket'].value_counts()

location_bucket
Canada    1269
Remote     132
             8
Name: count, dtype: int64

In [135]:
master.to_csv(r"/Users/pardeepwalia/Desktop/untitled folder/2025 datasets/master_2025.csv",index=False)